# Assignment 2: Encoders, and decoders
You will fine-tune two kinds of transformer and compare each model before and after training. Part A fine-tunes an encoder (DistilBERT) for named entity recognition. Part B turns a small decoder (SmolLM2-135M) into an assistant with supervised fine-tuning (SFT), then trains it with direct preference optimisation (DPO) to give shorter answers.

The training code is given. Your work is in the data, in a few functions in `src/` and in the cells marked `TODO`. Replace every `None` with what the comment after it asks for: a variable, a short expression or a short answer in quotes.

**Your data must be your own work.** Tasks 2 and 6 ask for data and judgements that only you can give: sentences from your own life, labels you decide on, and ratings of model outputs you have read. Do not ask an AI tool to write, label or rate them. Part II of Duo Quiz 2 includes questions about your own data, so check every item yourself.

**Submission.** Run all cells and save the notebook with its outputs. Running all cells again does not repeat training: each training cell loads the saved model and shows the loss plot saved with it. Zip the `assignment2` folder with the notebook, `src/` and `data/`, leaving out `.venv` and `runs/`. Upload the ZIP to the course platform. Late submissions follow the syllabus. Part II of Duo Quiz 2 asks about your own submission and multiplies your mark for this assignment.

| Task | Marks |
|---|---:|
| **Part A. Encoder** | |
| 1. Label alignment | 11 |
| 2. Your annotations | 22 |
| 3. Before and after | 10 |
| 4. Sniff tests | 8 |
| **Part B. Decoder** | |
| 5. Supervised fine-tuning | 12 |
| 6. Preference data | 17 |
| 7. DPO loss | 8 |
| 8. Before and after | 12 |
| **Total** | **100** |

## Setup
Run this cell at the start of every session. It installs the requirements. In Colab it first mounts Google Drive and moves to your copy of `assignment2`. Change the path if you cloned elsewhere. See the README for first-time setup.

In [ ]:
STUDENT_NUMBER = 1234567  # your student number, as a number
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    %cd /content/drive/MyDrive/cng463-assignments/assignment2
%pip install -q -r requirements.txt

In [ ]:
# Run this cell again after every edit to src/.
import importlib
import math
import os
from collections import Counter
import pandas as pd
import torch
from datasets import concatenate_datasets, load_dataset
from transformers import set_seed
import src.training, src.ner, src.preference
for module in (src.training, src.ner, src.preference): # reload the module to pick up any changes you made
    importlib.reload(module)
from src.training import DEVICE, free_memory
from src.ner import (LABELS, load_ner_model, align_labels, train_ner, predict_ner,
                     entity_f1, blank_annotation, read_annotation)
from src.preference import (MODEL as LM, load_lm, load_sft_data, train_sft, load_pref_data, read_pairs,
                            dpo_loss, train_dpo, generate, sequence_logprob)
set_seed(0)
print("Device:", DEVICE, "" if DEVICE != "cpu" else "(no GPU found, so training will be very slow)")

# Part A. Encoder: named entity recognition with DistilBERT
CoNLL-2003 labels each word of English news sentences with one of four entity types (PER, ORG, LOC, MISC) in the BIO scheme. DistilBERT splits words into subword tokens, but the labels belong to words.

## 1. Label alignment (11 marks)
Implement `align_labels` in `src/ner.py` (6). Show the alignment of a training sentence with at least one word split into subwords, and answer the questions (5).

In [ ]:
# TODO [Task 1]: Show the alignment of one training sentence.
conll = load_dataset("lhoestq/conll2003")  # columns tokens and ner_tags. Run "hf auth logout" if you get DatasetNotFoundError.
tokenizer, ner_model = load_ner_model()
example = conll["train"][None]  # use the index of a sentence with at least one word split into subwords
enc = tokenizer(example["tokens"], is_split_into_words=True)
aligned = align_labels(example["ner_tags"], enc.word_ids())
for token, word_id, label in zip(enc.tokens(), enc.word_ids(), aligned):
    print(f"{token:15} {str(word_id):5} {LABELS[label] if label != -100 else -100}")
print("\nWords:", None, " Tokens:", None)  # two expressions, one for the number of words in the sentence, and one for the number of tokens after tokenization
print("A word split into subwords:", None)  # the word, as in the sentence
print("Value of the loss at positions labelled -100:", None)  # a number

### How training works in this assignment
Every training cell works the same way.

* **Loss plot.** A plot under the cell shows the training loss and the validation loss, and it is redrawn while the model trains. The validation set is never trained on, so its loss shows how well the model does on new data. A training loss that keeps falling while the validation loss rises means the model is overfitting.
* **Early stopping.** After each evaluation, training stops if none of the last 3 evaluations reached a new lowest validation loss. For example, with validation losses 0.40, 0.31, 0.33, 0.32, 0.34, the lowest is 0.31 and the three evaluations after it are all higher, so training stops there. The rule is in `EarlyStopping` in `src/training.py`.
* **Checkpoints.** A checkpoint of the model and the optimiser is saved to `runs/` regularly, and each new checkpoint replaces the last one. If training is interrupted, run the cell again and it continues from the latest checkpoint.
* **Finished models.** When training ends, the final model is saved to `runs/`. Running the cell again loads that model instead of training again. Delete its folder in `runs/` to train from scratch.

Train the encoder. `untrained_ner` keeps the same model with a random classification layer for Task 3.

In [ ]:
# Fine-tune on CoNLL-2003. Do not change this cell.
untrained_ner = load_ner_model()[1]
ner_model = train_ner(ner_model, tokenizer, conll["train"], conll["validation"])

## 2. Your annotations (22 marks)
Write 15 English sentences from outside the news domain, taken from your own life, for example messages you sent, posts in a group or announcements you received. Include at least 15 entities, with each of PER, ORG, LOC and MISC at least once. Do not include other people's private details. Put the sentences in the next cell and run it once to create `data/ner_mine.tsv`. Then open the file and change the label of each entity word, following the CoNLL scheme: `B-` for the first word of an entity, `I-` for the words after it, and `O` for all other words. Your sentences (7) and your labels (15).

For example, the sentence `Elif Kaya posted on Instagram that METU NCC opens a new stadium.` is labelled as follows.

```
Elif	B-PER
Kaya	I-PER
posted	O
on	O
Instagram	B-ORG
that	O
METU	B-ORG
NCC	I-ORG
opens	O
a	O
new	O
stadium	O
.	O
```

*** In practice, more than one person labels the same data, and the labels they agree on become the gold standard. ***

In [ ]:
# Run once to create the file you label. It refuses to overwrite your labels.
MY_SENTENCES = [  # your 15 English sentences
]
blank_annotation(MY_SENTENCES, "data/ner_mine.tsv")  # one word and its label per line, as in CoNLL

In [ ]:
# Check your labelled file. Do not change this cell.
my_words, my_labels = read_annotation("data/ner_mine.tsv")
entities = Counter(label[2:] for s in my_labels for label in s if label.startswith("B-"))
print("Sentences:", len(my_words), " Words:", sum(map(len, my_words)), " Entities:", dict(entities))

## 3. Before and after (10 marks)
Compare the untrained and fine-tuned models on the first 500 CoNLL test sentences and on your own sentences, with your labels as the gold standard (8). Answer the questions (2).

In [ ]:
# TODO [Task 3]: Compare the two models in and out of domain.
test = conll["test"].select(range(500))
test_gold = [[LABELS[i] for i in tags] for tags in test["ner_tags"]]
rows = []
for name, model in [("untrained", untrained_ner), ("fine-tuned", ner_model)]:
    rows.append((name,
                 entity_f1(test_gold, predict_ner(model, tokenizer, test["tokens"])),  # entity F1 on the test sentences
                 entity_f1(my_labels, predict_ner(model, tokenizer, my_words))))      # entity F1 on your sentences
print(pd.DataFrame(rows, columns=["model", "CoNLL test F1", "your sentences F1"]).round(3).to_string(index=False))
my_pred = predict_ner(ner_model, tokenizer, my_words)
missed = Counter(g[2:] for gs, ps in zip(my_labels, my_pred) for g, p in zip(gs, ps) if g.startswith("B-") and p != g)
print("Drop in F1 out of domain:", round(None, 3))  # an expression: test F1 minus your F1, fine-tuned model
print("Entity type missed most often in your sentences:", None)  # PER, ORG, LOC or MISC

## 4. Sniff tests (8 marks)
A sniff test is a small input chosen to check one behaviour. Run the probes and add two of your own, each checking one behaviour (4). Name the probe the model fails most clearly (2) and the course concept that explains the failure, such as casing, subword tokenisation, domain shift or ambiguity (2).

In [ ]:
# TODO [Task 4]: Run the probes and add two of your own.
probes = ["apple opened a new store in istanbul .",
          "I ate an Apple on the way to Apple .",
          "Jordan flew to Jordan last week .",
          "Ghazaal Sheikhi moved from Famagusta to Güzelyurt .",
          "", # your probe: the same word as PER, ORG and LOC
          ""] # your probe: entity names in Turkish
for sentence in probes:
    words = sentence.split()
    print(list(zip(words, predict_ner(ner_model, tokenizer, [words])[0])))
print("\nProbe the model fails most clearly:", None)  # one probe, e.g. probes[2]
print("Course concept that explains it:", None)      # one course concept, in a few words

In [ ]:
# Part B needs most of the GPU memory, so this cell frees the modelss of Parts A. Do not change this cell.
# The model stays saved in runs/, so running its cells again loads it without training.
ner_model = untrained_ner = model = None
free_memory()

# Part B. Decoder: SFT and DPO with SmolLM2-135M
SmolLM2-135M is a base model. It continues text but has not learnt to follow instructions. SFT on instruction-response pairs teaches it the format. DPO then trains it on pairs of a chosen and a rejected answer. Here the chosen answers are short and well rated, so the target behaviour is short, direct answers.

## 5. Supervised fine-tuning (12 marks)
Implement `to_prompt_completion` in `src/preference.py` (6). Show one example and answer the questions (6).

In [ ]:
# TODO [Task 5]: Build the SFT data and inspect one example.
sft_data = load_sft_data()
sft_train, sft_val = sft_data.select(range(2000)), sft_data.select(range(2000, len(sft_data)))
print(sft_train[None]["prompt"] + sft_train[None]["completion"]) # an index of a training example
print("\nTraining examples:", None)  # an expression: the number of examples in sft_train
print("Mean completion length in words:", round(sum(len(c.split()) for c in sft_train["completion"]) / len(sft_train), 1))  # a number
print("Part the SFT loss is computed on:", None)  # "prompt", "completion" or "both"
print("Token that train_sft adds to every completion:", None)  # the token, as text

In [ ]:
# SFT on 2,000 examples. The model is saved to runs/sft. Do not change this cell.
lm_tok, lm = load_lm()
train_sft(lm, lm_tok, sft_train, sft_val)

## 6. Preference data (17 marks)
`load_pref_data` takes pairs from UltraFeedback in which the chosen answer has at most 120 words and the rejected one is at least 1.5 times as long. 50 pairs are kept for validation during training and the last 50 for Task 8.

Run the next cell once with your student number as the seed. It writes 10 pairs to `data/pref_audit.csv`. Read each pair yourself. Fill in `chosen_better` with `yes` or `no` and `reason` with one of `shorter`, `more_accurate`, `follows_instruction` or `unclear` (5). For example, for the prompt `Name three primary colours.`, the chosen answer `Red, yellow and blue.` and a rejected answer that lists the same colours in a long paragraph about colour mixing, `chosen_better` is `yes` and `reason` is `shorter`.

Write 10 pairs of your own in `data/my_pairs.jsonl`, one JSON object per line, like this example (10).

```
{"prompt": "What is the boiling point of water at sea level?", "chosen": "100 °C, or 212 °F.", "rejected": "At sea level, water boils at 100 °C, or 212 °F. The boiling point depends on air pressure, so it is lower at high altitudes, for example about 95 °C in a mountain city."}
```

Use questions you really asked this term, of a person or a search engine, and write both answers yourself. Make the chosen answer short and correct, and the rejected answer correct but longer than needed. Answer the questions (2).

In [ ]:
# Run once to create the audit file. It refuses to overwrite your labels.
pref = load_pref_data()
n = len(pref)
pref_train, pref_val, pref_held = pref.select(range(n - 100)), pref.select(range(n - 100, n - 50)), pref.select(range(n - 50, n))
if os.path.exists("data/pref_audit.csv"):
    print("data/pref_audit.csv exists. Delete it first if you want to start again.")
else:
    sample = pref_train.shuffle(seed=STUDENT_NUMBER).select(range(10))
    pd.DataFrame({"prompt": sample["prompt"], "chosen": sample["chosen"], "rejected": sample["rejected"],
                  "chosen_better": "", "reason": ""}).to_csv("data/pref_audit.csv", index=False)

In [ ]:
# TODO [Task 6]: Describe the preference data.
my_pairs = read_pairs("data/my_pairs.jsonl")
train_pairs = concatenate_datasets([pref_train, my_pairs])
pref_audit = pd.read_csv("data/pref_audit.csv")
print(pref_audit[["chosen_better", "reason"]].value_counts().to_string())
chosen_words = [len(c.split()) for c in train_pairs["chosen"]]
rejected_words = [len(r.split()) for r in train_pairs["rejected"]]
print("\nProvided pairs:", None, " Your pairs:", None)  # two expressions: the number of provided pairs and the number of your pairs
print("Mean words, chosen and rejected:", round(None, 1),
      round(None, 1))  # two expressions: the mean number of words in chosen and rejected
print("Share of training pairs where chosen is shorter:",
      round(sum(c < r for c, r in zip(chosen_words, rejected_words)) / len(chosen_words), 2)) 

## 7. DPO loss (8 marks)
For a prompt x, a chosen answer y_w and a rejected answer y_l, DPO minimises

−log σ(β [(log π(y_w|x) − log π_ref(y_w|x)) − (log π(y_l|x) − log π_ref(y_l|x))])

where π is the model being trained and π_ref is the frozen SFT model. Implement `dpo_loss` in `src/preference.py` (6). Check it on the three cases below with β = 0.1 and answer the question (2).

In [ ]:
# TODO [Task 7]: Check dpo_loss on three cases.
t = torch.tensor
print("Policy equals reference:", dpo_loss(t([-10.]), t([-12.]), t([-10.]), t([-12.])).item())  # Policy and reference both give -10 (chosen) and -12 (rejected)
print("Policy raises chosen, lowers rejected by 5 each:", dpo_loss(t([-5.]), t([-17.]), t([-10.]), t([-12.])).item())  # Policy -5 and -17, reference -10 and -12
print("Policy lowers chosen, raises rejected by 5 each:", dpo_loss(t([-15.]), t([-7.]), t([-10.]), t([-12.])).item())  # Policy -15 and -7, reference -10 and -12
print("Exact value when the policy equals the reference:", None)  # an exact expression in quotes, not a decimal

In [ ]:
# DPO from the SFT model. The model is saved to runs/dpo. Do not change this cell.
# Watch rewards/accuracies and rewards/margins in the log.
lm_tok, lm = load_lm("runs/sft")
train_dpo(lm, lm_tok, train_pairs, pref_val)

## 8. Before and after (12 marks)
Compare the base, SFT and DPO models on ten held-out instructions two of them your own (4). Compute the DPO loss on the 50 held-out pairs with the SFT model as the reference (4). Run the sniff test, which checks whether DPO changed behaviour outside its target, and answer the questions (4).

In [ ]:
# TODO [Task 8]: Compare the three models.
HELD_OUT = ["Explain what a CPU does.", "How do I boil an egg?", "What is the capital of Australia?",
            "Give me one tip for learning a new language.", "Why is the sky blue?", "What is photosynthesis?",
            "How can I sleep better?", "What does HTTP stand for?",
            "", ""]  # your two instructions
STORY = "Write a 200-word story about a cat."
rows, answers, models = [], {}, {}
for name, path in [("base", LM), ("SFT", "runs/sft"), ("DPO", "runs/dpo")]:
    lm_tok, models[name] = load_lm(path)
    answers[name] = generate(models[name], lm_tok, HELD_OUT + [STORY], max_new_tokens=300)
    held = answers[name][:len(HELD_OUT)]
    rows.append((name, sum(len(a.split()) for a, _ in held) / len(held),  # mean words per answer, over HELD_OUT only
                 sum(stopped for _, stopped in held) / len(held)))        # share of answers that stopped, over HELD_OUT only
    print(f"{name}: {answers[name][0][0]}\n")
print(pd.DataFrame(rows, columns=["model", "mean words", "stopped"]).round(2).to_string(index=False))

logp = {name: t([[sequence_logprob(models[name], lm_tok, r["prompt"], r[k]) for k in ("chosen", "rejected")]
                 for r in pref_held]) for name in ("SFT", "DPO")}
held_loss = dpo_loss(logp["DPO"][:, 0], logp["DPO"][:, 1], logp["SFT"][:, 0], logp["SFT"][:, 1]).item()
print("\nHeld-out DPO loss:", round(None, 3))  # a variable
print("Held-out loss below the value when the policy equals the reference:", None if held_loss < math.log(2) else None)  # "yes" or "no"
print("Story length in words, SFT and DPO:", len(None), len(None))  # two expressions: the number of words in the story generated by SFT and DPO
print("Did DPO change behaviour outside its target?", None)  # "yes" or "no"